# Lab 09 — CSV, Parquet, and HDFS

**Tujuan:** Mengamati format file, tipe data, column pruning, dan HDFS.

**Lingkungan:** Spark standalone (`spark-master:7077`), Jupyter, dua worker. Jalankan sel secara berurutan. Setiap notebook dapat dijalankan sendiri.

**Metode:** Prediksi → jalankan → amati `explain()`/Spark UI → ubah satu variabel → simpulkan.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Spark Training - Lab 09")
    .master("spark://spark-master:7077")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "4")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)

## 1. Generate an independent training dataset

Dataset disiapkan di dalam notebook; tidak membutuhkan file CSV yang sudah ada. Penulisan memakai shared volume agar seluruh executor dapat membaca.

In [ ]:
from pathlib import Path

base = "/home/spark/outputs/lab09"
df = spark.range(0, 100_000, 1, 4)
data = df.withColumn("customer_id", F.col("id") % 1000)
data = data.withColumn("amount", (F.col("id") % 5000) + 100)
data = data.withColumn("month", (F.col("id") % 12) + 1)
data.write.mode("overwrite").option("header", True).csv(base + "/csv")
data.write.mode("overwrite").parquet(base + "/parquet")

## 2. Read both formats

In [ ]:
csv_df = spark.read.csv(base + "/csv", header=True, inferSchema=True)
parquet_df = spark.read.parquet(base + "/parquet")
csv_df.printSchema()
parquet_df.printSchema()

## 3. Compare scan plans

Cari `ReadSchema` dan `PushedFilters` pada physical scan.

In [ ]:
csv_df.select("customer_id", "amount").explain("formatted")
parquet_df.select("customer_id", "amount").explain("formatted")

## 4. Optional HDFS write/read

Memakai NameNode yang Anda sudah jalankan. Jika permission atau HDFS connectivity belum tersedia, cukup gunakan bagian file lokal di atas; tidak perlu mengubah Compose.

In [ ]:
hdfs_path = "hdfs://namenode:8020/tmp/spark-training/lab09/parquet"
try:
    data.write.mode("overwrite").parquet(hdfs_path)
    hdfs_df = spark.read.parquet(hdfs_path)
    print("HDFS rows:", hdfs_df.count())
except Exception as error:
    print("HDFS demonstration unavailable:", str(error)[:500])

**Tugas:** jelaskan bedanya file part (Spark output) dengan logical partitions, dan mengapa `SELECT` sedikit kolom lebih menguntungkan pada Parquet.

## Refleksi peserta

Tuliskan: **(1)** apa prediksi Anda, **(2)** apa yang teramati di Spark UI atau output, **(3)** mengapa Spark berperilaku demikian. Hindari menyimpulkan dari waktu eksekusi satu kali saja.